# 量化神经网络

该模块实现了量化神经网络(Quantized Neural Network，QNN)的模拟运算，包括量化和反量化。这些算子允许在不改变数据类型的情况下模拟 QNN 的输出，支持动态数据类型选择以及通道级和标量级的缩放因子和零点参数。

In [1]:
from tvm.topi.nn import qnn

In [2]:
qnn.simulated_quantize?

Signature:
qnn.simulated_quantize(
    data,
    out_dtype,
    output_scale=None,
    output_zero_point=None,
    axis=-1,
)
Docstring:
Simulated QNN quantize operator that mimics QNN outputs without changing datatype.
The benefit of this operator over true QNN quantize is that this operator allows dynamic
datatype selection and can operate on both per-channel and scalar scales and zero points while
QNN quantize requires both of these to be fixed at compile time.

Parameters
----------
data: tvm.te.Tensor
    An N-D input tensor to the operator.

out_dtype: tvm.te.Tensor
    A scalar variable that indicates which datatype to simulate quantization with. Use
    SQNN_DTYPE_TO_CODE to convert a dtype string into the corresponding variable
    value.

output_scale: tvm.te.Tensor, optional
    A scalar tensor representing the scale to use when quantizing to integer datatypes.
    When it contains more than a single value, N must match the number of channels in data.

output_zero_point: tvm

{func}`~tvm.topi.nn.qnn.simulated_quantize` 模拟 QNN 量化算子，在不改变数据类型的情况下模拟 QNN 输出。该算子相比真实 QNN 量化的优势在于支持动态数据类型选择，并且可以同时处理通道级和标量级的缩放因子和零点参数，而真实 QNN 量化要求这些参数在编译时固定。

In [3]:
qnn.simulated_dequantize?

Signature:
qnn.simulated_dequantize(
    data,
    in_dtype,
    input_scale=None,
    input_zero_point=None,
    axis=-1,
)
Docstring:
Simulated QNN dequantize operator that mimics QNN outputs without changing datatype.
The benefit of this operator over true QNN dequantize is that this operator allows dynamic
datatype selection and can operate on both per-channel and scalar scales and zero points while
QNN dequantize requires both of these to be fixed at compile time.

Parameters
----------
data: tvm.te.Tensor
    An N-D input tensor to the operator.

in_dtype: tvm.te.Tensor
    A scalar variable that indicates which datatype to simulate dequantization with. Use
    SQNN_DTYPE_TO_CODE to convert a dtype string into the corresponding variable
    value.

input_scale: tvm.te.Tensor, optional
    A scalar tensor representing the scale to use when dequantizing from integer datatypes.
    When it contains more than a single value, N must match the number of channels in data.

input_zero_po

{func}`~tvm.topi.nn.qnn.simulated_dequantize` 模拟 QNN 反量化算子，在不改变数据类型的情况下模拟 QNN 输出。该算子相比真实 QNN 反量化的优势在于支持动态数据类型选择，并且可以同时处理通道级和标量级的缩放因子和零点参数，而真实 QNN 反量化要求这些参数在编译时固定。

## 量化和反量化函数参数说明

以下是 `simulated_quantize` 和 `simulated_dequantize` 函数的共同参数和差异参数说明

- 共同参数:
1. `data: tvm.te.Tensor` - 输入的 N 维张量
2. `axis: int, optional` - 量化/反量化的通道轴，默认值为 `-1`（对应最后一个轴）

- 差异参数:
    - 量化函数(simulated_quantize):
    - `out_dtype: tvm.te.Tensor` - 标量变量，表示要模拟的量化数据类型
    - `output_scale: tvm.te.Tensor, optional` - 标量张量，表示量化到整数数据类型时使用的缩放因子
    - `output_zero_point: tvm.te.Tensor, optional` - 1维张量，表示量化到整数数据类型时使用的零点

    - 反量化函数(simulated_dequantize):
    - `in_dtype: tvm.te.Tensor` - 标量变量，表示要模拟的反量化数据类型
    - `input_scale: tvm.te.Tensor, optional` - 标量张量，表示从整数数据类型反量化时使用的缩放因子
    - `input_zero_point: tvm.te.Tensor, optional` - 1维张量，表示从整数数据类型反量化时使用的零点

模拟任意整数数据类型的量化。所有数据类型的计算方式为：
```python
Q_output = clip(
    (round(input_tensor/output_scale) + output_zero_point),
    out_dtype::min, 
    out_dtype::max
)
```

模拟任意整数数据类型的反量化。所有数据类型的计算方式为：
```python
DQ_output = (input - zero_point) * scale
```

示例：

In [4]:
import tvm
from tvm import te, relax
from tvm.relax.testing import nn

In [ ]:
from tvm_book.op import register
op_name = "qnn.simulated_quantize"
reg = register.Register(op_name)
reg.describe(
    R"code(Simulates the functionality of qnn.quantize but allows more flexible"
    "dynamic input type conversion and always outputs float values.)code"
)
tvm.ir.Op.get(op_name).add_argument("data", "Tensor", "The tensor to quantize.")
tvm.ir.Op.get(op_name).add_argument("out_dtype", "Tensor", "A code corresponding to the type of quantization to apply.")
tvm.ir.Op.get(op_name).add_argument("output_scale", "Tensor", "The quantization scale of the output tensor.")
tvm.ir.Op.get(op_name).add_argument("output_zero_point", "Tensor", "The quantization zero_point of the output tensor.")

In [ ]:
def infer_struct_info(call: relax.Call, context):
    activations, weight, bias = call.args

    matmul_call = relax.op.matmul(activations, weight)
    matmul_sinfo = tvm.ir.Op.get("relax.matmul").get_attr("FInferStructInfo")(
        matmul_call, context
    )

    matmul_var = relax.Var("dummy_var", matmul_sinfo)
    add_call = matmul_var + bias
    add_sinfo = tvm.ir.Op.get("relax.add").get_attr("FInferStructInfo")(add_call, context)
    return add_sinfo

In [ ]:
auto output_sinfo = make_object<TensorStructInfoNode>(*input_sinfo.get());
  output_sinfo->dtype = attrs->out_dtype;

In [40]:
bb = relax.BlockBuilder()

# 1. 创建输入张量
data = nn.Placeholder((1, 3, 224, 224), dtype="float32", name="data")
# 2. 定义量化参数
scale = nn.Placeholder((3,), dtype="float32", name='scale')
zero_point = nn.Placeholder((3,), dtype="float32", name='zero_point')
in_dtype = nn.Placeholder((), dtype="int32", name='in_dtype') 
out_dtype = nn.Placeholder((), dtype="int32", name='out_dtype')
with bb.function("main", [data, out_dtype, scale, zero_point]):
    # 3. 应用量化
    with bb.dataflow():
        quantized = bb.call_te(qnn.simulated_quantize, data, out_dtype, scale, zero_point, axis=1, primfunc_name_hint="simulated_quantize",)
        # dequantized = bb.call_te(qnn.simulated_dequantize, quantized, in_dtype, scale, zero_point, axis=1, primfunc_name_hint="simulated_quantize",)
        data = nn.Placeholder((1, 3, 224, 224), dtype="float32", name="data")
        scale = nn.Placeholder((3,), dtype="float32", name='scale')
        zero_point = nn.Placeholder((3,), dtype="float32", name='zero_point')
        dtype_code = relax.const(qnn.SQNN_INT8, 'int32')
        out = relax.Call(tvm.ir.Op.get(op_name), args=[data, dtype_code, scale, zero_point])
        gv = bb.emit_output([quantized, out])
    bb.emit_func_output(gv)
mod = bb.get()
# mod = relax.transform.AnnotateTIROpPattern()(mod)
# mod = relax.transform.FuseOps()(mod)

InternalError: Check failed: (op_map_infer_struct_info_.count(op)) is false:  Cannot find the FInferStructInfo attribute registered to op: qnn.simulated_quantize

In [38]:
mod.show()